<a href="https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Basil-Maqbool/flyrank-internship-assignment1"
REPO_DIR = "flyrank-internship-assignment1"

if IN_COLAB:
    # Clone the repo if it isn't already here
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

    # Navigate to the correct notebooks folder
    target_dir = f"{REPO_DIR}/work/notebooks"
    if os.path.basename(os.getcwd()) != "notebooks":
        os.chdir(target_dir)

print("Current Working Directory:", os.getcwd())

Current Working Directory: /content/flyrank-internship-assignment1/work/notebooks


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

If a page has not been updated in over 180 days (Stale) but still receives over 500 impressions (Visible), it receives a maximum baseline priority score of 100. If it is over 90 days old with some traffic, it gets a score of 50. Everything else gets 0.

Reason Codes:

stale_visible_page (Score 100): High priority. Page is aging but still ranking/getting traffic.

aging_page (Score 50): Medium priority. Approaching stale threshold.

no_action_needed (Score 0): Low priority.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import pandas as pd
import numpy as np
import os

# 1. Load the dataset (using the starter CSV for the baseline)
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# 0. Define the target proxy
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

# 2. Encode the Rule
def calculate_baseline_score(row):
    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        return 100
    elif row['days_since_last_update'] >= 90 and row['impressions_90d'] >= 100:
        return 50
    return 0

df['baseline_score'] = df.apply(calculate_baseline_score, axis=1)

# 3. Add Reason Codes and Action Labels
df['reason_code'] = np.where(df['baseline_score'] == 100, 'stale_visible_page',
                    np.where(df['baseline_score'] == 50, 'aging_page', 'no_action_needed'))
df['action_label'] = np.where(df['baseline_score'] > 0, 'Review for Content Refresh', 'Monitor')

# 4. Sort to create the ranked queue (highest score and highest impressions first)
ranked_queue = df.sort_values(by=['baseline_score', 'impressions_90d'], ascending=[False, False])

# 5. Save to CSV
os.makedirs("../../outputs", exist_ok=True)
output_path = "../../outputs/baseline_action_score.csv"
ranked_queue[['content_id', 'baseline_score', 'reason_code', 'action_label']].to_csv(output_path, index=False)

print(f"Queue saved to {output_path}")

# Display top 20 for the next step
display(ranked_queue[['content_id', 'days_since_last_update', 'impressions_90d', 'baseline_score', 'reason_code']].head(20))

# 6. Evaluate: Precision@K and Base Rate (required by building-baselines skill)
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df['is_declining_label'].mean()
p50 = precision_at_k(df['baseline_score'], df['is_declining_label'], 50)
p20 = precision_at_k(df['baseline_score'], df['is_declining_label'], 20)

print("\n=== BASELINE PERFORMANCE ===")
print(f"Base Rate (Majority Class): {base_rate:.2f}")
print(f"Baseline Precision@20:      {p20:.2f}")
print(f"Baseline Precision@50:      {p50:.2f}")
print(f"\nPrecision@50 of {p50:.2f} vs base rate of {base_rate:.2f}: the baseline {'beats' if p50 > base_rate else 'does not beat'} random picking.")

Queue saved to ../../outputs/baseline_action_score.csv


,content_id,days_since_last_update,impressions_90d,baseline_score,reason_code
16751,content_cf56e2e2e282,194,61678,100,stale_visible_page
16514,content_7368877ea310,194,59472,100,stale_visible_page
7021,content_1bfaa38ff26c,194,25715,100,stale_visible_page
21268,content_0a91db491d14,193,13299,100,stale_visible_page
11489,content_5feee3994adb,194,7812,100,stale_visible_page
12045,content_c2d929d83eaa,193,7558,100,stale_visible_page
698,content_b16bd7307b39,194,4590,100,stale_visible_page
5327,content_fe16a55cd13d,194,4556,100,stale_visible_page
26810,content_ecb6215e79fd,194,4429,100,stale_visible_page
20837,content_928af3e22c80,193,1697,100,stale_visible_page


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Rows 1-5: Action: Content Refresh | Code: stale_visible_page | Note: Very high traffic. What makes it wrong: These might be static "Contact Us" or "About" pages that never need refreshing despite high impressions.

Rows 6-10: Action: Content Refresh | Code: stale_visible_page | Note: Strong traffic. What makes it wrong: The traffic could be heavily seasonal (e.g., summer-specific content) rather than structural decline, meaning a rewrite wouldn't help.

Rows 11-15: Action: Content Refresh | Code: stale_visible_page | Note: Moderate traffic. What makes it wrong: These pages might currently hold Position #1 on Google. Touching them could break their momentum and actually hurt visibility.

Rows 16-20: Action: Content Refresh | Code: stale_visible_page | Note: Moderate traffic. What makes it wrong: The query intent might be fully satisfied by a short answer. Adding more content just because it is "stale" might lower the user experience.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks Analysis:
The weakest picks in this baseline are pages that score 100 purely based on age, but are already perfectly fulfilling their search intent. A rigid rule cannot tell the difference between a 200-day-old news article (needs updating) and a 200-day-old mathematical formula (never needs updating).

Leakage Check:
Confirmed. No future-window data was used, and no FlyRank product flags (like health_score, priority_score, or trend_pct) leaked into the baseline logic. The score relies entirely on observable, historical data: days_since_last_update and impressions_90d.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.